# Semana 7: Redes Recurrentes (RNN, LSTM, GRU) para NLP
## Notebook Conceptual (NB1) – RNN y LSTM con Datos Sintéticos

**Propósito:** Modelar secuencias de texto capturando dependencias temporales mediante redes recurrentes.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Implementar una RNN simple desde cero y visualizar el estado oculto.
- Demostrar el problema del gradiente desvaneciente con secuencias largas.
- Construir una LSTM con PyTorch y comparar su capacidad para recordar información lejana.
- Entrenar una LSTM para clasificación de secuencias.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y configuramos PyTorch.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Configuración de PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Semillas para reproducibilidad
np.random.seed(42)
torch.manual_seed(42)

print("\nLibrerías importadas correctamente.")

---
## 1. Datos Sintéticos: Secuencias de Números

Generamos secuencias de números para dos tareas:
1. **Predicción del siguiente número**: Secuencias como [1,2,3,4,?]
2. **Clasificación de secuencias**: Determinar si una secuencia es creciente o no.

In [ ]:
# Tarea 1: Predecir el siguiente número en una secuencia aritmética
def generate_arithmetic_sequences(n_samples=1000, seq_len=5):
    """Genera secuencias aritméticas: [a, a+d, a+2d, ..., a+(seq_len-1)d]"""
    X = []
    y = []
    for _ in range(n_samples):
        a = np.random.randint(1, 10)
        d = np.random.randint(1, 5)
        seq = [a + i*d for i in range(seq_len)]
        X.append(seq[:-1])  # primeros 4 números
        y.append(seq[-1])    # último número
    return np.array(X), np.array(y)

X_arith, y_arith = generate_arithmetic_sequences(1000, 5)
print("Ejemplo de secuencia aritmética:")
print(f"X: {X_arith[0]}, y: {y_arith[0]}")

# Tarea 2: Clasificar si una secuencia es creciente o no
def generate_sequence_classification(n_samples=1000, seq_len=10):
    """Genera secuencias y etiquetas: 1 si es creciente, 0 si no"""
    X = []
    y = []
    for _ in range(n_samples):
        if np.random.random() > 0.5:
            # Secuencia creciente
            start = np.random.randint(1, 10)
            step = np.random.randint(1, 3)
            seq = [start + i*step for i in range(seq_len)]
            label = 1
        else:
            # Secuencia aleatoria (no creciente)
            seq = np.random.randint(1, 20, seq_len).tolist()
            label = 0
        X.append(seq)
        y.append(label)
    return np.array(X), np.array(y)

X_class, y_class = generate_sequence_classification(1000, 10)
print("\nEjemplo de clasificación de secuencias:")
print(f"X: {X_class[0]}, Creciente: {bool(y_class[0])}")

---
## 2. Implementación de una RNN Simple desde Cero

Implementamos una RNN manualmente para entender el flujo del estado oculto.

In [ ]:
class SimpleRNNManual:
    """Implementación manual de una RNN para un paso."""
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        # Inicializar pesos aleatoriamente
        self.Wxh = np.random.randn(hidden_size, input_size) * 0.01
        self.Whh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.bh = np.zeros((hidden_size, 1))
    
    def forward(self, x, h_prev):
        """
        x: entrada en el paso actual (input_size, 1)
        h_prev: estado oculto anterior (hidden_size, 1)
        """
        h = np.tanh(self.Wxh @ x + self.Whh @ h_prev + self.bh)
        return h

# Ejemplo con datos pequeños
input_size = 1
hidden_size = 3
rnn = SimpleRNNManual(input_size, hidden_size)

# Secuencia de ejemplo
seq = [1, 2, 3, 4]
h = np.zeros((hidden_size, 1))

print("Evolución del estado oculto:")
for i, x_val in enumerate(seq):
    x = np.array([[x_val]])
    h = rnn.forward(x, h)
    print(f"Paso {i+1}, entrada={x_val}: h = {h.flatten().round(3)}")

### 2.1. RNN con PyTorch

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, hidden = self.rnn(x)
        # Usamos la salida del último paso
        out = self.fc(out[:, -1, :])
        return out

# Ejemplo con datos de clasificación
input_size = 1
hidden_size = 10
output_size = 2  # binario

model_rnn = SimpleRNN(input_size, hidden_size, output_size)

# Preparar datos para PyTorch
X_class_t = torch.tensor(X_class, dtype=torch.float32).unsqueeze(-1)  # (n_samples, seq_len, 1)
y_class_t = torch.tensor(y_class, dtype=torch.long)

dataset = TensorDataset(X_class_t, y_class_t)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print("Modelo RNN creado.")

---
## 3. Demostración del Problema del Gradiente Desvaneciente

Simulamos el cálculo del gradiente en una RNN para una secuencia larga y mostramos cómo se desvanece.

In [ ]:
def simulate_vanishing_gradient(seq_len, hidden_size=10):
    """Simula la magnitud del gradiente en RNN para diferentes longitudes."""
    # Pesos aleatorios
    Whh = np.random.randn(hidden_size, hidden_size) * 0.1
    
    # Gradiente inicial
    grad = np.ones((hidden_size, 1))
    grad_norms = []
    
    for t in range(seq_len):
        grad = Whh.T @ grad
        grad_norms.append(np.linalg.norm(grad))
    
    return grad_norms

seq_lengths = [5, 10, 20, 50, 100]
plt.figure(figsize=(10, 5))

for seq_len in seq_lengths:
    norms = simulate_vanishing_gradient(seq_len)
    plt.plot(range(1, seq_len+1), norms, label=f'seq_len={seq_len}')

plt.xlabel('Paso de tiempo')
plt.ylabel('Norma del gradiente')
plt.title('Simulación del gradiente desvaneciente en RNN')
plt.legend()
plt.yscale('log')
plt.grid(True)
plt.show()

print("Observación: La norma del gradiente decae exponencialmente con la longitud de la secuencia.")

---
## 4. LSTM con PyTorch

Implementamos un modelo LSTM para clasificación de secuencias.

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, (hidden, cell) = self.lstm(x)
        # Usamos el último estado oculto
        out = self.fc(hidden[-1])
        return out

model_lstm = LSTMModel(input_size=1, hidden_size=10, output_size=2, num_layers=1)
print("Modelo LSTM creado.")

---
## 5. Entrenamiento Comparativo: RNN vs LSTM

Entrenamos ambos modelos en la tarea de clasificación de secuencias y comparamos su rendimiento.

In [ ]:
def train_model(model, dataloader, epochs=20, lr=0.001):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    losses = []
    accuracies = []
    
    for epoch in range(epochs):
        epoch_loss = 0
        correct = 0
        total = 0
        
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
        
        epoch_loss /= len(dataloader)
        epoch_acc = correct / total
        losses.append(epoch_loss)
        accuracies.append(epoch_acc)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")
    
    return losses, accuracies

print("Entrenando RNN...")
losses_rnn, accs_rnn = train_model(model_rnn, dataloader, epochs=30)

print("\nEntrenando LSTM...")
losses_lstm, accs_lstm = train_model(model_lstm, dataloader, epochs=30)

### 5.1. Visualización comparativa

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(accs_rnn, label='RNN')
axes[0].plot(accs_lstm, label='LSTM')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Comparación de Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(losses_rnn, label='RNN')
axes[1].plot(losses_lstm, label='LSTM')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Pérdida')
axes[1].set_title('Comparación de Pérdida')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

---
## 6. Experimento: Memoria a Largo Plazo

Creamos un problema donde la información relevante está al inicio de una secuencia larga, y comparamos RNN vs LSTM.

In [ ]:
def generate_long_term_dependency_data(n_samples=500, seq_len=20, delay=15):
    """
    Genera secuencias donde el primer dígito indica la clase,
    pero hay muchos pasos irrelevantes después.
    """
    X = []
    y = []
    for _ in range(n_samples):
        # El primer elemento determina la clase (1 o 0)
        first = np.random.randint(0, 2)
        # El resto son ruido
        rest = np.random.randint(0, 5, seq_len-1)
        seq = np.concatenate([[first], rest])
        X.append(seq)
        y.append(first)
    return np.array(X), np.array(y)

X_long, y_long = generate_long_term_dependency_data(500, 20)

# Preparar datos para PyTorch
X_long_t = torch.tensor(X_long, dtype=torch.float32).unsqueeze(-1)
y_long_t = torch.tensor(y_long, dtype=torch.long)

dataset_long = TensorDataset(X_long_t, y_long_t)
loader_long = DataLoader(dataset_long, batch_size=32, shuffle=True)

# Crear modelos
model_rnn_long = SimpleRNN(input_size=1, hidden_size=20, output_size=2)
model_lstm_long = LSTMModel(input_size=1, hidden_size=20, output_size=2)

print("Entrenando en tarea de memoria a largo plazo...")
print("\nRNN:")
losses_rnn_long, accs_rnn_long = train_model(model_rnn_long, loader_long, epochs=30)
print("\nLSTM:")
losses_lstm_long, accs_lstm_long = train_model(model_lstm_long, loader_long, epochs=30)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(accs_rnn_long, label='RNN')
plt.plot(accs_lstm_long, label='LSTM')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.title('Memoria a Largo Plazo: RNN vs LSTM')
plt.legend()
plt.grid(True)
plt.show()

print(f"Accuracy final RNN: {accs_rnn_long[-1]:.4f}")
print(f"Accuracy final LSTM: {accs_lstm_long[-1]:.4f}")

---
## 7. GRU (Gated Recurrent Unit)

Implementamos un modelo GRU y comparamos con LSTM.

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        out, hidden = self.gru(x)
        out = self.fc(hidden[-1])
        return out

model_gru = GRUModel(input_size=1, hidden_size=20, output_size=2)
print("Modelo GRU creado.")

# Entrenar GRU en la tarea de memoria
print("\nEntrenando GRU...")
losses_gru_long, accs_gru_long = train_model(model_gru, loader_long, epochs=30)

# Comparación final
plt.figure(figsize=(10, 5))
plt.plot(accs_rnn_long, label='RNN')
plt.plot(accs_lstm_long, label='LSTM')
plt.plot(accs_gru_long, label='GRU')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.title('Comparación: RNN vs LSTM vs GRU')
plt.legend()
plt.grid(True)
plt.show()

---
## 8. Ejercicio para el Estudiante

1. **Predicción del siguiente número**: Entrena una LSTM para predecir el siguiente número en secuencias aritméticas (datos `X_arith`, `y_arith`).
2. **Experimenta con la longitud de secuencia**: Genera secuencias más largas (ej. 50 pasos) y observa cómo se comportan RNN vs LSTM.
3. **Bidireccional**: Implementa una LSTM bidireccional y compara con la unidireccional.
4. **Visualización de puertas**: Si te sientes aventurero, intenta visualizar los valores de las puertas de la LSTM para una secuencia.

In [ ]:
# Espacio para el estudiante
# ...

---
## 9. Conclusiones

En este notebook hemos explorado las redes recurrentes para modelado de secuencias:

✔️ **RNN simple**: Implementación manual y con PyTorch.

✔️ **Gradiente desvaneciente**: Demostramos cómo el gradiente decae con la longitud de la secuencia.

✔️ **LSTM**: Arquitectura con puertas que mitiga el gradiente desvaneciente.

✔️ **Comparación RNN vs LSTM**: LSTM supera a RNN en tareas que requieren memoria a largo plazo.

✔️ **GRU**: Alternativa más simple con rendimiento similar.

**Lección clave**: Para tareas con dependencias de largo alcance, LSTM y GRU son muy superiores a RNN vanilla. Sin embargo, todas son secuenciales y han sido superadas por Transformers en muchas aplicaciones.

En el próximo notebook (NB2) aplicaremos LSTM a un problema real de clasificación de sentimiento.

---
**Fin del Notebook Conceptual - Semana 7 (NLP)**